In [53]:
%%writefile pathfinding.py
import pygame
import math
import heapq
import time
import random

pygame.init()

# ================= SETTINGS =================
WIDTH = 600
ROWS = 30
INFO_HEIGHT = 80

WIN = pygame.display.set_mode((WIDTH, WIDTH + INFO_HEIGHT))
pygame.display.set_caption("A* and GBFS")

WHITE = (255,255,255)
BLACK = (0,0,0)
RED = (255,0,0)
GREEN = (0,255,0)
BLUE = (0,0,255)
GREY = (200,200,200)

FONT = pygame.font.SysFont("Arial",18)

current_algorithm = "None"
nodes_visited = 0
path_cost = -1
exec_time = 0
heuristic_mode = "Manhattan"

# ================= NODE =================
class Node:
    def __init__(self,row,col,width,total_rows):
        self.row=row
        self.col=col
        self.x=col*width
        self.y=row*width
        self.color=WHITE
        self.neighbors=[]
        self.width=width
        self.total_rows=total_rows

    def get_pos(self):
        return self.row,self.col

    def is_barrier(self):
        return self.color==BLACK

    def reset(self):
        self.color=WHITE

    def make_start(self):
        self.color=GREEN

    def make_end(self):
        self.color=RED

    def make_barrier(self):
        self.color=BLACK

    def make_path(self):
        self.color=BLUE

    def draw(self,win):
        pygame.draw.rect(
            win,
            self.color,
            (self.x,self.y+INFO_HEIGHT,self.width,self.width)
        )

    def update_neighbors(self,grid):
        self.neighbors=[]
        if self.row<self.total_rows-1 and not grid[self.row+1][self.col].is_barrier():
            self.neighbors.append(grid[self.row+1][self.col])
        if self.row>0 and not grid[self.row-1][self.col].is_barrier():
            self.neighbors.append(grid[self.row-1][self.col])
        if self.col<self.total_rows-1 and not grid[self.row][self.col+1].is_barrier():
            self.neighbors.append(grid[self.row][self.col+1])
        if self.col>0 and not grid[self.row][self.col-1].is_barrier():
            self.neighbors.append(grid[self.row][self.col-1])

    def __lt__(self,other):
        return False


# ================= HEURISTICS =================
def heuristic(a,b):
    x1,y1=a.get_pos()
    x2,y2=b.get_pos()

    if heuristic_mode=="Manhattan":
        return abs(x1-x2)+abs(y1-y2)
    else:
        return math.sqrt((x1-x2)**2+(y1-y2)**2)


# ================= GRID =================
def make_grid(rows,width):
    grid=[]
    gap=width//rows

    for i in range(rows):
        grid.append([])
        for j in range(rows):
            node=Node(i,j,gap,rows)
            grid[i].append(node)

    # random obstacles
    for row in grid:
        for node in row:
            if random.random()<0.25:
                node.make_barrier()

    return grid


def draw_grid(win,rows,width):
    gap=width//rows
    for i in range(rows):
        pygame.draw.line(win,GREY,(0,i*gap+INFO_HEIGHT),(width,i*gap+INFO_HEIGHT))
        pygame.draw.line(win,GREY,(i*gap,INFO_HEIGHT),(i*gap,width+INFO_HEIGHT))


# ================= DRAW =================
def draw(win,grid):
    win.fill(WHITE)

    pygame.draw.rect(win,(220,220,220),(0,0,WIDTH,INFO_HEIGHT))

    info1=FONT.render(f"Algorithm: {current_algorithm}",True,BLACK)
    info2=FONT.render(f"Nodes Visited: {nodes_visited}",True,BLACK)
    info3=FONT.render(f"Path Cost: {path_cost}",True,BLACK)
    info4=FONT.render(f"Execution Time: {exec_time} ms",True,BLACK)
    info5=FONT.render(f"Heuristic: {heuristic_mode}",True,BLACK)

    win.blit(info1,(10,10))
    win.blit(info2,(10,25))
    win.blit(info3,(10,40))
    win.blit(info4,(10,55))
    win.blit(info5,(300,25))

    for row in grid:
        for node in row:
            node.draw(win)

    draw_grid(win,ROWS,WIDTH)
    pygame.display.update()


# ================= PATH =================
def reconstruct_path(came_from,current):
    global path_cost
    cost=0
    while current in came_from:
        current=came_from[current]
        current.make_path()
        cost+=1
    path_cost=cost


# ================= A* =================
def astar(grid,start,end):
    global current_algorithm,nodes_visited,exec_time
    current_algorithm="A*"
    nodes_visited=0

    start_time=time.time()

    count=0
    open_set=[]
    heapq.heappush(open_set,(0,count,start))

    came_from={}
    g_score={node:float("inf") for row in grid for node in row}
    g_score[start]=0

    f_score={node:float("inf") for row in grid for node in row}
    f_score[start]=heuristic(start,end)

    open_hash={start}

    while open_set:
        current=open_set[0][2]
        heapq.heappop(open_set)
        open_hash.remove(current)

        nodes_visited+=1

        if current==end:
            reconstruct_path(came_from,end)
            exec_time=int((time.time()-start_time)*1000)
            return True

        for neighbor in current.neighbors:
            temp_g=g_score[current]+1

            if temp_g<g_score[neighbor]:
                came_from[neighbor]=current
                g_score[neighbor]=temp_g
                f_score[neighbor]=temp_g+heuristic(neighbor,end)

                if neighbor not in open_hash:
                    count+=1
                    heapq.heappush(open_set,(f_score[neighbor],count,neighbor))
                    open_hash.add(neighbor)

    exec_time=int((time.time()-start_time)*1000)
    return False


# ================= GBFS =================
def gbfs(grid,start,end):
    global current_algorithm,nodes_visited,exec_time
    current_algorithm="GBFS"
    nodes_visited=0

    start_time=time.time()

    open_set=[]
    count=0
    heapq.heappush(open_set,(0,count,start))

    came_from={}
    visited=set()

    while open_set:
        current=heapq.heappop(open_set)[2]
        nodes_visited+=1

        if current==end:
            reconstruct_path(came_from,end)
            exec_time=int((time.time()-start_time)*1000)
            return True

        visited.add(current)

        for neighbor in current.neighbors:
            if neighbor not in visited:
                count+=1
                came_from[neighbor]=current
                heapq.heappush(open_set,(heuristic(neighbor,end),count,neighbor))

    exec_time=int((time.time()-start_time)*1000)
    return False

# ================= CLEAR OLD PATH =================
def clear_path(grid, start, end):
    global path_cost
    path_cost = -1

    for row in grid:
        for node in row:
            if node != start and node != end:
                if node.color == BLUE:
                    node.reset()
                    
# ================= RESET FUNCTION =================
def reset_grid():
    global nodes_visited, path_cost, exec_time, current_algorithm

    grid = make_grid(ROWS, WIDTH)

    start = grid[0][0]
    end = grid[ROWS-1][ROWS-1]

    start.make_start()
    end.make_end()

    nodes_visited = 0
    path_cost = -1
    exec_time = 0
    current_algorithm = "None"

    return grid, start, end


# ================= UTIL =================
def get_clicked_pos(pos,rows,width):
    gap=width//rows
    y,x=pos
    y=y-INFO_HEIGHT
    if y<0:
        return None,None
    row=y//gap
    col=x//gap
    return row,col


# ================= MAIN =================
def main():
    grid, start, end = reset_grid()

    run=True
    while run:
        draw(WIN,grid)

        for event in pygame.event.get():
            if event.type==pygame.QUIT:
                run=False

            if event.type==pygame.KEYDOWN:

                for row in grid:
                    for node in row:
                        node.update_neighbors(grid)

                if event.key==pygame.K_a:
                    clear_path(grid, start, end)
                    astar(grid,start,end)

                if event.key==pygame.K_g:
                    clear_path(grid, start, end)
                    gbfs(grid,start,end)

                if event.key==pygame.K_m:
                    global heuristic_mode
                    heuristic_mode="Manhattan"

                if event.key==pygame.K_e:
                    heuristic_mode="Euclidean"

                if event.key==pygame.K_r:
                    grid, start, end = reset_grid()

    pygame.quit()


main()

Overwriting pathfinding.py


In [54]:
!python pathfinding.py

pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


C:\Users\DELL\anaconda3\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
